In [2]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
# from fastapi import FastAPI
from llama_index.core import StorageContext, load_index_from_storage
from custom_llm import OurLLM
from llama_index.core import ServiceContext, set_global_service_context
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import VectorStoreIndex, get_response_synthesizer
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.postprocessor import SimilarityPostprocessor
from llama_index.core.query_engine import CustomQueryEngine
from llama_index.core.retrievers import BaseRetriever
from llama_index.core import get_response_synthesizer
from llama_index.core.response_synthesizers import BaseSynthesizer

llmR = OurLLM()
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
set_global_service_context(ServiceContext.from_defaults(llm=llmR, embed_model=embed_model, chunk_size=1024, context_window = 2048, num_output=256))


storage_context = StorageContext.from_defaults(persist_dir="mini512")
index = load_index_from_storage(storage_context)

C:\Users\M LAKSHMI NARASIMHA\AppData\Local\Temp\ipykernel_20804\228564415.py:18: DeprecationWarning: Call to deprecated class method from_defaults. (ServiceContext is deprecated, please use `llama_index.settings.Settings` instead.) -- Deprecated since version 0.10.0.
  set_global_service_context(ServiceContext.from_defaults(llm=llmR, embed_model=embed_model, chunk_size=1024, context_window = 2048, num_output=256))


In [2]:
base_retriever = index.as_retriever(similarity_top_k=5)

In [3]:
from llama_index.llms.openai import OpenAI
from llama_index.core import PromptTemplate

qa_prompt = PromptTemplate(
    "Context information is below.\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "Given the context information and not prior knowledge, "
    "answer the query.\n"
    "Query: {query_str}\n"
    "Answer: "
)


class RAGStringQueryEngine(CustomQueryEngine):
    """RAG String Query Engine."""

    retriever: BaseRetriever
    response_synthesizer: BaseSynthesizer
    llm: OurLLM
    qa_prompt: PromptTemplate

    def custom_query(self, query_str: str):
        nodes = self.retriever.retrieve(query_str)

        context_str = "\n\n".join([n.node.get_content() for n in nodes])
        response = self.llm.complete(
            qa_prompt.format(context_str=context_str, query_str=query_str)
        )

        return str(response)

query_engine = RAGStringQueryEngine(
    retriever=base_retriever,
    response_synthesizer=get_response_synthesizer(),
    llm=llmR,
    qa_prompt=qa_prompt,
)

In [4]:
response = query_engine.query("""Describe the conversation between hanuma and sita""") 
from IPython.display import display, Markdown, Latex
display(Markdown(response.response))

Prompt:  Context information is below.
---------------------
And restored to confidence, the
        two began to converse with one another. (7)
        Hearing the aforesaid reply of Sita,
        who was stricken with grief, Hanuman, son
        of the wind-god, endeavoured to draw nearer
        to her. (8)
        The nearer did the celebrated Hanuman
        draw to her, the more did that Sita suspect
        him to be Ravana. (9)
        She said to herself : ‘Oh, shame | What
        a pity that | held all this converse with him.
        Indeed, he is the same Ravana, who has
        come here assuming another guise.’ (10)
        
        
        Letting go that branch of the Asoka
        tree, Sita, for her part, of faultless limbs,
        sank down on that very spot on which she
        stood, exhausted as she was from grief.
        
        (11)
        
        Thereupon Hanuman of mighty arms
        saluted the aforesaid daughter of Janaka.
        Sore stricken as sh

The conversation between Hanuman and Sita is marked by suspicion, fear, and ultimately, a glimmer of hope.

Initially, Sita suspects Hanuman to be Ravana in disguise. She is terrified, having already endured a previous encounter where Ravana presented himself as a humble hermit. She finds it difficult to trust anyone and believes Hanuman to be toying with her emotions. 

Hanuman, on the other hand, approaches Sita cautiously and respectfully. Aware of her fragile mental state, he chooses his words carefully, aiming to comfort her without alarming her further. He sings praises of Rama, hoping to gain her trust. 

Sita, initially skeptical, starts to listen as Hanuman reveals details about Rama that only someone close to him would know. This sparks a flicker of hope within her. She questions Hanuman, scrutinizing his every word for any hint of deceit.

Over time, Sita's fear gives way to cautious optimism as she becomes convinced of Hanuman's genuineness and his connection to Rama.  The conversation ends with Sita feeling a surge of hope and praying for the truth of Hanuman's words. 
